In [ ]:
import os
import random
import logging
import datetime
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import tensorflow as tf
import keras
from tensorflow.keras import layers, mixed_precision
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau, TensorBoard, ModelCheckpoint, CSVLogger)
from tensorflow.keras.losses import CategoricalCrossentropy
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)
from sklearn.utils import shuffle, class_weight as sk_class_weight

warnings.filterwarnings('ignore')
logging.getLogger('tensorflow').setLevel(logging.ERROR)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

if gpus:
    mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision: float16 enabled")
else:
    print("No GPU detected — running on CPU (mixed precision disabled)")

print(f"\nTensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print(f"GPUs       : {gpus or 'None'}")

In [ ]:
@dataclass
class Config:
    train_dir : Path = Path('/content/drive/MyDrive/project2/final_dataset/train')
    val_dir   : Path = Path('/content/drive/MyDrive/project2/final_dataset/val')
    test_dir  : Path = Path('/content/drive/MyDrive/project2/final_dataset/test')
    output_dir: Path = Path('outputs')

    image_size   : int = 224
    num_channels : int = 3

    dense_units    : int   = 256
    dropout_rate1  : float = 0.50
    dropout_rate2  : float = 0.30
    label_smoothing: float = 0.10

    p1_epochs  : int   = 50
    p1_lr      : float = 1e-3
    batch_size : int   = 32

    p2_epochs        : int   = 100
    p2_lr            : float = 1e-4
    fine_tune_layers : int   = 60

    early_stopping_patience : int   = 12
    reduce_lr_patience      : int   = 4
    reduce_lr_factor        : float = 0.3
    min_lr                  : float = 1e-8

    seed : int = 42

    num_classes  : int       = field(init=False, default=0)
    class_names  : list      = field(init=False, default_factory=list)
    timestamp    : str       = field(init=False,
                                     default_factory=lambda: datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))

    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)
        (self.output_dir / 'checkpoints').mkdir(exist_ok=True)
        (self.output_dir / 'logs').mkdir(exist_ok=True)
        (self.output_dir / 'figures').mkdir(exist_ok=True)

        self.class_names = sorted(
            d.name for d in self.train_dir.iterdir() if d.is_dir()
        )
        self.num_classes = len(self.class_names)

        random.seed(self.seed)
        np.random.seed(self.seed)
        tf.random.set_seed(self.seed)

    @property
    def checkpoint_path(self) -> Path:
        return self.output_dir / 'checkpoints' / 'best_model.keras'

    @property
    def log_dir(self) -> Path:
        return self.output_dir / 'logs' / self.timestamp

    @property
    def input_shape(self) -> tuple:
        return (self.image_size, self.image_size, self.num_channels)


cfg = Config()

print(f"Classes ({cfg.num_classes}): {cfg.class_names}")
print(f"Input shape            : {cfg.input_shape}")
print(f"Output directory       : {cfg.output_dir.resolve()}")

In [ ]:
from concurrent.futures import ThreadPoolExecutor

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'}


def load_single_image(args):
    fpath, image_size = args
    img = cv2.imread(str(fpath))
    if img is None:
        return None, None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
    return img, str(fpath.name)


def load_split(directory: Path, cfg: Config):
    images, labels = [], []
    class_to_idx = {name: idx for idx, name in enumerate(cfg.class_names)}

    for class_name, class_idx in class_to_idx.items():
        class_dir = directory / class_name
        files = [f for f in class_dir.iterdir() if f.suffix.lower() in VALID_EXTENSIONS]
        args  = [(f, cfg.image_size) for f in files]

        with ThreadPoolExecutor(max_workers=8) as executor:
            results = list(tqdm(
                executor.map(load_single_image, args),
                total=len(files),
                desc=f"  {class_name:30s}",
                leave=False
            ))

        for (img, fname) in results:
            if img is None:
                continue
            images.append(img)
            labels.append(class_idx)

    X = np.array(images, dtype=np.uint8)
    y = np.array(labels, dtype=np.int32)
    return shuffle(X, y, random_state=cfg.seed)


X_train, y_train = load_split(cfg.train_dir, cfg)
X_val,   y_val   = load_split(cfg.val_dir,   cfg)
X_test,  y_test  = load_split(cfg.test_dir,  cfg)

print(f"\n{'Split':8s} {'Shape':25s} {'N':>6s}")
print("-" * 42)
for name, X, y in [('Train', X_train, y_train), ('Val', X_val, y_val), ('Test', X_test, y_test)]:
    print(f"{name:8s} {str(X.shape):25s} {len(X):>6d}")

In [ ]:
class_weights_array = sk_class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights_array))

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

@tf.function
def augment(image: tf.Tensor, label: tf.Tensor):
    image = tf.cast(image, tf.float32)

    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    k     = tf.random.uniform([], 0, 4, dtype=tf.int32)
    image = tf.image.rot90(image, k)

    image = tf.image.random_brightness(image, max_delta=0.2 * 255)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    image = tf.image.random_hue(image, max_delta=0.05)
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label


@tf.function
def cast_to_float(image: tf.Tensor, label: tf.Tensor):
    image = tf.cast(image, tf.float32)
    return image, label

def make_dataset(
    X: np.ndarray,
    y: np.ndarray,
    cfg: Config,
    training: bool = False,
) -> tf.data.Dataset:
    y_ohe = tf.keras.utils.to_categorical(y, num_classes=cfg.num_classes)
    ds = tf.data.Dataset.from_tensor_slices((X, y_ohe))

    if training:
        ds = ds.shuffle(buffer_size=len(X), seed=cfg.seed, reshuffle_each_iteration=True)
        ds = ds.map(augment,         num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(cast_to_float,   num_parallel_calls=AUTOTUNE)

    return ds.batch(cfg.batch_size).prefetch(AUTOTUNE)


ds_train = make_dataset(X_train, y_train, cfg, training=True)
ds_val   = make_dataset(X_val,   y_val,   cfg, training=False)
ds_test  = make_dataset(X_test,  y_test,  cfg, training=False)

print(f"Train batches : {len(ds_train)}")
print(f"Val batches   : {len(ds_val)}")
print(f"Test batches  : {len(ds_test)}")

In [ ]:
def plot_sample_grid(X: np.ndarray, y: np.ndarray, cfg: Config, n_per_class: int = 3) -> None:
    rng       = np.random.default_rng(cfg.seed)
    n_cls     = cfg.num_classes
    fig, axes = plt.subplots(n_cls, n_per_class,
                             figsize=(n_per_class * 2.8, n_cls * 2.8))
    if n_cls == 1:
        axes = axes[np.newaxis, :]

    for row, (cls_idx, cls_name) in enumerate(enumerate(cfg.class_names)):
        pool   = np.where(y == cls_idx)[0]
        chosen = rng.choice(pool, size=min(n_per_class, len(pool)), replace=False)
        for col, img_idx in enumerate(chosen):
            axes[row, col].imshow(X[img_idx])
            axes[row, col].axis('off')
            if col == 0:
                axes[row, col].set_ylabel(cls_name, fontsize=9, fontweight='bold',
                                          rotation=0, labelpad=70, va='center')
        for col in range(len(chosen), n_per_class):
            axes[row, col].axis('off')

    fig.suptitle('Training Set — Sample Images per Class',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(cfg.output_dir / 'figures' / 'sample_grid.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_sample_grid(X_train, y_train, cfg)

In [ ]:
def plot_class_distribution(splits: dict, cfg: Config) -> None:
    counts = {}
    for split_name, y in splits.items():
        unique, c = np.unique(y, return_counts=True)
        counts[split_name] = dict(zip(unique, c))

    df = pd.DataFrame(counts, index=range(cfg.num_classes))
    df.index = cfg.class_names

    fig, ax = plt.subplots(figsize=(max(10, cfg.num_classes * 0.8), 5))
    df.plot(kind='bar', ax=ax, width=0.7,
            color=['steelblue', 'darkorange', 'forestgreen'])
    ax.set_xlabel('Class', fontweight='bold')
    ax.set_ylabel('Sample Count', fontweight='bold')
    ax.set_title('Class Distribution Across Splits', fontsize=13, fontweight='bold')
    ax.legend(frameon=False)
    plt.xticks(rotation=45, ha='right')
    sns.despine(ax=ax)
    plt.tight_layout()
    plt.savefig(cfg.output_dir / 'figures' / 'class_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_class_distribution({'Train': y_train, 'Val': y_val, 'Test': y_test}, cfg)

In [ ]:
def plot_augmentation_preview(X: np.ndarray, cfg: Config, n_examples: int = 5) -> None:
    rng  = np.random.default_rng(cfg.seed)
    idxs = rng.choice(len(X), size=n_examples, replace=False)

    fig, axes = plt.subplots(2, n_examples, figsize=(n_examples * 3, 6))
    axes[0, 0].set_ylabel('Original',  fontweight='bold', fontsize=10)
    axes[1, 0].set_ylabel('Augmented', fontweight='bold', fontsize=10)

    for col, idx in enumerate(idxs):
        orig = X[idx]
        aug, _ = augment(
            tf.constant(orig[np.newaxis], dtype=tf.uint8),
            tf.constant([0])
        )
        axes[0, col].imshow(orig)
        axes[0, col].axis('off')
        axes[1, col].imshow(aug[0].numpy().astype(np.uint8))
        axes[1, col].axis('off')

    fig.suptitle('Augmentation Preview', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(cfg.output_dir / 'figures' / 'augmentation_preview.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_augmentation_preview(X_train, cfg)

In [ ]:
def build_model(cfg: Config, trainable_backbone: bool = False) -> Model:
    backbone = EfficientNetV2B0(
        include_top=False,
        weights='imagenet',
        input_shape=cfg.input_shape,
        include_preprocessing=True,
    )

    backbone.trainable = trainable_backbone

    if trainable_backbone:
        for layer in backbone.layers[:-cfg.fine_tune_layers]:
            layer.trainable = False

        for layer in backbone.layers:
            if isinstance(layer, layers.BatchNormalization):
                layer.trainable = False

    inputs = backbone.input
    x      = backbone.output

    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.Dropout(cfg.dropout_rate1, name='drop1')(x)
    x = layers.Dense(
        cfg.dense_units,
        activation='gelu',
        kernel_initializer='he_normal',
        kernel_regularizer=tf.keras.regularizers.l2(1e-4),
        name='dense_head',
    )(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.Dropout(cfg.dropout_rate2, name='drop2')(x)
    x = layers.Activation('linear', dtype='float32', name='cast_fp32')(x)

    outputs = layers.Dense(
        cfg.num_classes,
        activation='softmax',
        dtype='float32',
        name='predictions',
    )(x)

    return Model(inputs=inputs, outputs=outputs, name='ParasiteNet')

def compile_model(model: Model, learning_rate: float, cfg: Config) -> None:
    model.compile(
        optimizer=AdamW(
            learning_rate=learning_rate,
            weight_decay=1e-5,
            epsilon=1e-7,
        ),
        loss=CategoricalCrossentropy(label_smoothing=cfg.label_smoothing),
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc', multi_label=False),
            tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc'),
        ],
    )


model = build_model(cfg, trainable_backbone=False)
compile_model(model, cfg.p1_lr, cfg)
model.summary(line_length=100, expand_nested=False)

total_params     = model.count_params()
trainable_params = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f"\nTotal params     : {total_params:>12,}")
print(f"Trainable params : {trainable_params:>12,} ({trainable_params / total_params * 100:.1f}%)")

In [ ]:
def make_callbacks(cfg: Config, phase: int) -> list:
    ckpt_path = cfg.output_dir / 'checkpoints' / f'phase{phase}_best.keras'
    csv_path  = cfg.output_dir / 'logs'         / f'phase{phase}_history.csv'

    return [
        EarlyStopping(
            monitor='val_auc',
            patience=cfg.early_stopping_patience,
            mode='max',
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor='val_auc',
            factor=cfg.reduce_lr_factor,
            patience=cfg.reduce_lr_patience,
            min_lr=cfg.min_lr,
            mode='max',
            verbose=1,
        ),
        ModelCheckpoint(
            filepath=str(ckpt_path),
            monitor='val_auc',
            save_best_only=True,
            mode='max',
            verbose=1,
        ),
        CSVLogger(str(csv_path), separator=',', append=False),
        TensorBoard(
            log_dir=str(cfg.log_dir / f'phase{phase}'),
            histogram_freq=1,
            update_freq='epoch',
        ),
    ]

In [ ]:
print("─" * 60)
print(" Phase 1: Training classification head (backbone frozen)")
print("─" * 60)

history_p1 = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=100,
    callbacks=make_callbacks(cfg, phase=1),
    class_weight=class_weights if USE_CLASS_WEIGHTS else None,
    verbose=1,
)

In [ ]:
def unfreeze_top_layers(model: Model, cfg: Config) -> None:
    for layer in model.layers:
        layer.trainable = False

    for layer in model.layers[-cfg.fine_tune_layers:]:
        layer.trainable = True

    bn_count = 0
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
            bn_count += 1

    trainable_count = sum(l.trainable for l in model.layers)
    print(f"Total layers     : {len(model.layers)}")
    print(f"Trainable layers : {trainable_count}")
    print(f"BN frozen        : {bn_count}")
    print("─" * 60)


print("─" * 60)
print(" Phase 2: Fine-tuning top backbone layers")
print("─" * 60)

unfreeze_top_layers(model, cfg)

compile_model(model, cfg.p2_lr, cfg)

trainable_params = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f"Trainable params (P2): {trainable_params:,}")

def unfreeze_top_layers(model: Model, cfg: Config) -> None:
    for layer in model.layers:
        layer.trainable = False

    for layer in model.layers[-cfg.fine_tune_layers:]:
        layer.trainable = True

    bn_count = 0
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
            bn_count += 1

    trainable_count = sum(l.trainable for l in model.layers)
    print(f"Total layers     : {len(model.layers)}")
    print(f"Trainable layers : {trainable_count}")
    print(f"BN frozen        : {bn_count}")
    print("─" * 60)


print("─" * 60)
print(" Phase 2: Fine-tuning top backbone layers")
print("─" * 60)

unfreeze_top_layers(model, cfg)
compile_model(model, cfg.p2_lr, cfg)

trainable_params = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f"Trainable params (P2): {trainable_params:,}")

history_p2 = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=100,
    callbacks=make_callbacks(cfg, phase=2),
    class_weight=class_weights if USE_CLASS_WEIGHTS else None,
    verbose=1,
)

In [ ]:
model.save('parasitenet_final.h5')

In [ ]:
def plot_training_curves(
    h1: keras.callbacks.History,
    h2: Optional[keras.callbacks.History] = None,
) -> None:

    merged = {}

    for key, vals in h1.history.items():
        merged[key] = vals

    if h2:
        for key, vals in h2.history.items():
            merged[key] = merged.get(key, []) + vals

    phase_boundary = len(h1.history['accuracy'])
    epochs = list(range(1, len(merged['accuracy']) + 1))

    colors_dark = ["#1F1F1F", "#313131", "#636363", "#AEAEAE", "#DADADA"]

    colors_red = [
        "#331313",
        "#582626",
        "#9E1717",
        "#D35151",
        "#E9B4B4",
    ]

    colors_green = [
        "#01411C",
        "#4B6F44",
        "#4F7942",
        "#74C365",
        "#D0F0C0",
    ]

    fig, ax = plt.subplots(1, 3, figsize=(20, 6))

    fig.text(
        s='Epochs vs. Training and Validation Metrics',
        size=20,
        fontweight='bold',
        fontname='monospace',
        color=colors_dark[1],
        y=1.03,
        x=0.25,
        alpha=0.9
    )

    metrics = [
        ('accuracy', 'val_accuracy', 'Accuracy'),
        ('loss', 'val_loss', 'Loss'),
        ('auc', 'val_auc', 'AUC'),
    ]

    for axis, (train_key, val_key, title) in zip(ax, metrics):

        if train_key not in merged:
            continue

        axis.plot(
            epochs,
            merged[train_key],
            marker='o',
            markersize=5,
            markerfacecolor=colors_green[2],
            color=colors_green[3],
            linewidth=2.5,
            label=f'Training {title}'
        )

        if val_key in merged:
            axis.plot(
                epochs,
                merged[val_key],
                marker='o',
                markersize=5,
                markerfacecolor=colors_red[2],
                color=colors_red[3],
                linewidth=2.5,
                linestyle='--',
                label=f'Validation {title}'
            )

        if h2:
            axis.axvline(
                phase_boundary,
                color=colors_dark[2],
                linestyle=':',
                linewidth=2,
                label='Fine-tune Start'
            )

        axis.set_title(
            title,
            fontsize=16,
            fontweight='bold',
            color=colors_dark[0]
        )

        axis.set_xlabel(
            'Epochs',
            fontsize=12,
            color=colors_dark[1]
        )

        axis.set_ylabel(
            title,
            fontsize=12,
            color=colors_dark[1]
        )

        axis.legend(frameon=False, fontsize=10)

        axis.tick_params(axis='both', labelsize=10)

        if title in ['Accuracy', 'AUC']:
            axis.set_ylim(0, 1.05)

        sns.despine(ax=axis)

    plt.tight_layout()
    plt.show()

plot_training_curves(history_p1, history_p2)

In [ ]:
def run_evaluation(
    model: Model,
    ds: tf.data.Dataset,
    y_int: np.ndarray,
    cfg: Config,
    split_name: str = 'Test',
) -> dict:
    metrics      = model.evaluate(ds, verbose=0)
    metric_names = model.metrics_names

    probs  = model.predict(ds, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    y_true = y_int

    y_true_ohe = tf.keras.utils.to_categorical(y_true, cfg.num_classes)
    macro_auc  = roc_auc_score(y_true_ohe, probs, average='macro', multi_class='ovr')

    print(f"\n{'─'*60}")
    print(f" {split_name} Results")
    print(f"{'─'*60}")
    for name, val in zip(metric_names, metrics):
        print(f"  {name:20s}: {val:.4f}")
    print(f"  {'macro_auc':20s}: {macro_auc:.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=cfg.class_names, digits=4))

    return {
        'loss'    : metrics[0],
        'accuracy': metrics[1],
        'auc'     : macro_auc,
        'y_true'  : y_true,
        'y_pred'  : y_pred,
        'probs'   : probs,
    }


results = run_evaluation(model, ds_test, y_test, cfg)

In [ ]:
def plot_confusion_matrix(results: dict, cfg: Config, normalise: bool = True) -> None:
    y_true, y_pred = results['y_true'], results['y_pred']
    cm_raw  = confusion_matrix(y_true, y_pred)
    cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2 if normalise else 1,
                             figsize=(14 if normalise else 8, 10))
    if not normalise:
        axes = [axes]

    pairs = [('Counts', cm_raw, 'd', 'Blues')]
    if normalise:
        pairs.append(('Recall (row-normalised)', cm_norm, '.2f', 'Greens'))

    for ax, (title, cm, fmt, cmap) in zip(axes, pairs):
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cfg.class_names)
        disp.plot(ax=ax, cmap=cmap, colorbar=True, xticks_rotation='vertical', values_format=fmt)
        ax.set_title(title, fontweight='bold', pad=12)

    fig.suptitle('Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(cfg.output_dir / 'figures' / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_confusion_matrix(results, cfg)

In [ ]:
def plot_per_class_metrics(results: dict, cfg: Config) -> None:
    from sklearn.metrics import precision_recall_fscore_support

    precision, recall, f1, support = precision_recall_fscore_support(
        results['y_true'], results['y_pred'], labels=range(cfg.num_classes)
    )

    df = pd.DataFrame({
        'Class'    : cfg.class_names,
        'Precision': precision,
        'Recall'   : recall,
        'F1 Score' : f1,
        'Support'  : support,
    }).sort_values('F1 Score', ascending=True)

    macro_f1 = f1.mean()

    fig, axes = plt.subplots(1, 3,
                             figsize=(16, max(5, cfg.num_classes * 0.5 + 1)),
                             sharey=True)

    for ax, metric in zip(axes, ['Precision', 'Recall', 'F1 Score']):
        colours = ['#D9534F' if v < macro_f1 else '#5CB85C' for v in df[metric]]
        bars = ax.barh(df['Class'], df[metric], color=colours)
        ax.axvline(macro_f1, color='steelblue', ls='--', lw=1.5, label=f'Macro avg ({macro_f1:.3f})')
        ax.set_title(metric, fontweight='bold')
        ax.set_xlim(0, 1.05)
        ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
        ax.legend(frameon=False, fontsize=8)
        sns.despine(ax=ax)

    fig.suptitle('Per-Class Metrics — Test Set  (red = below macro avg)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(cfg.output_dir / 'figures' / 'per_class_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))


plot_per_class_metrics(results, cfg)

In [ ]:
class GradCAM:
    def __init__(self, model: Model, last_conv_layer_name: str):
        self.grad_model = Model(
            inputs=model.inputs,
            outputs=[
                model.get_layer(last_conv_layer_name).output,
                model.output,
            ],
        )

    def __call__(self, image: np.ndarray, class_idx: Optional[int] = None) -> np.ndarray:
        tensor = tf.convert_to_tensor(image[np.newaxis], dtype=tf.float32)

        with tf.GradientTape() as tape:
            tape.watch(tensor)
            conv_outputs, predictions = self.grad_model(tensor, training=False)
            if class_idx is None:
                class_idx = int(tf.argmax(predictions[0]))
            target_score = predictions[:, class_idx]

        grads   = tape.gradient(target_score, conv_outputs)
        pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
        heatmap = tf.reduce_sum(conv_outputs[0] * pooled, axis=-1)
        heatmap = tf.nn.relu(heatmap).numpy()

        if heatmap.max() > 0:
            heatmap /= heatmap.max()
        return heatmap

    @staticmethod
    def overlay(
        image: np.ndarray,
        heatmap: np.ndarray,
        alpha: float = 0.45,
        colormap: int = cv2.COLORMAP_JET,
    ) -> np.ndarray:
        h, w = image.shape[:2]
        heatmap_u8   = np.uint8(255 * heatmap)
        heatmap_resz = cv2.resize(heatmap_u8, (w, h))
        coloured     = cv2.applyColorMap(heatmap_resz, colormap)
        coloured     = cv2.cvtColor(coloured, cv2.COLOR_BGR2RGB)
        return np.uint8((1 - alpha) * image + alpha * coloured)

last_conv = next(
    l.name for l in reversed(model.layers)
    if isinstance(l, (tf.keras.layers.Conv2D,
                      tf.keras.layers.DepthwiseConv2D,
                      tf.keras.layers.Activation))
    and len(l.output.shape) == 4
)
print(f"Last conv-like layer: {last_conv}")
gcam = GradCAM(model, last_conv_layer_name=last_conv)

In [ ]:
def plot_gradcam_grid(
    model: Model,
    gcam: GradCAM,
    X: np.ndarray,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    cfg: Config,
    n_samples: int = 12,
    n_cols: int = 4,
) -> None:
    rng     = np.random.default_rng(cfg.seed)
    indices = rng.choice(len(X), size=min(n_samples, len(X)), replace=False)
    n_rows  = (len(indices) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows * 2, n_cols,
                             figsize=(n_cols * 3, n_rows * 6))

    for i, idx in enumerate(indices):
        row_orig = (i // n_cols) * 2
        row_cam  = row_orig + 1
        col      = i % n_cols

        img     = X[idx]
        img_f32 = img.astype(np.float32)
        heatmap = gcam(img_f32, class_idx=int(y_pred[idx]))
        blended = GradCAM.overlay(img, heatmap)

        true_name = cfg.class_names[y_true[idx]]
        pred_name = cfg.class_names[y_pred[idx]]
        colour    = 'green' if true_name == pred_name else 'red'
        label     = f"T: {true_name}\nP: {pred_name}"

        axes[row_orig, col].imshow(img)
        axes[row_orig, col].set_title(label, fontsize=7.5, color=colour, fontweight='bold')
        axes[row_orig, col].axis('off')
        axes[row_cam,  col].imshow(blended)
        axes[row_cam,  col].set_title('Grad-CAM', fontsize=7.5, color='grey')
        axes[row_cam,  col].axis('off')

    for ax in axes.flatten()[len(indices) * 2:]:
        ax.axis('off')

    fig.suptitle('Grad-CAM Visualisations  (green = correct, red = incorrect)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(cfg.output_dir / 'figures' / 'gradcam.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_gradcam_grid(model, gcam, X_test, results['y_true'], results['y_pred'], cfg)

In [ ]:
def predict(
    image_source,
    model: Model,
    cfg: Config,
    top_k: int = 3,
    show: bool = True,
) -> dict:

    if isinstance(image_source, str):
        path = Path(image_source)
        if not path.exists():
            raise FileNotFoundError(f"Image not found: {path}")
        img_bgr = cv2.imread(str(path))
        if img_bgr is None:
            raise ValueError(f"Could not decode image: {path}")
        img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    else:
        img = image_source.copy()

    img_resized = cv2.resize(img, (cfg.image_size, cfg.image_size), interpolation=cv2.INTER_LINEAR)

    tensor = tf.constant(img_resized[np.newaxis], dtype=tf.float32)

    probs    = model.predict(tensor, verbose=0)[0]
    top_idxs = np.argsort(probs)[::-1][:top_k]
    pred_idx = int(top_idxs[0])

    result = {
        'predicted_class'  : cfg.class_names[pred_idx],
        'confidence'       : float(probs[pred_idx]),
        'top_k'            : [
            {'class': cfg.class_names[i], 'probability': float(probs[i])}
            for i in top_idxs
        ],
        'all_probabilities': dict(zip(cfg.class_names, probs.tolist())),
    }

    if show:
        fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(9, 4),
                                             gridspec_kw={'width_ratios': [1, 1.5]})
        ax_img.imshow(img)
        ax_img.set_title(
            f"{result['predicted_class']}\n({result['confidence']:.1%})",
            fontweight='bold', fontsize=11
        )
        ax_img.axis('off')

        top_classes = [d['class'] for d in result['top_k']]
        top_probs   = [d['probability'] for d in result['top_k']]
        bars = ax_bar.barh(top_classes[::-1], top_probs[::-1],
                           color=['#2196F3' if i == 0 else '#90CAF9'
                                  for i in range(len(top_classes) - 1, -1, -1)])
        ax_bar.bar_label(bars, fmt='%.3f', padding=3)
        ax_bar.set_xlim(0, 1.1)
        ax_bar.set_title(f'Top-{top_k} Predictions', fontweight='bold')
        ax_bar.set_xlabel('Probability')
        sns.despine(ax=ax_bar)
        plt.tight_layout()
        plt.show()

    return result

for cls_dir in sorted(cfg.test_dir.iterdir()):
    if cls_dir.is_dir():
        files = [f for f in cls_dir.iterdir() if f.suffix.lower() in VALID_EXTENSIONS]
        if files:
            result = predict(str(files[0]), loaded_model, cfg)
            print(f"Ground truth : {cls_dir.name}")
            print(f"Prediction   : {result['predicted_class']}  ({result['confidence']:.1%})")
            break

In [ ]:
summary = {
    'Model'       : 'ParasiteNet (EfficientNetV2B0)',
    'Input size'  : f"{cfg.image_size}×{cfg.image_size}",
    'Classes'     : cfg.num_classes,
    'Test acc.'   : f"{results['accuracy']*100:.2f}%",
    'Macro AUC'   : f"{results['auc']:.4f}",
    'Final model' : str(final_model_path.resolve()),
    'Figures'     : str((cfg.output_dir / 'figures').resolve()),
}

pad = max(len(k) for k in summary)
print("\n" + "═" * 55)
print(" ParasiteNet — Final Results")
print("═" * 55)
for k, v in summary.items():
    print(f"  {k:{pad}s} : {v}")
print("═" * 55)